# BariatricRSD Data Exploration

This notebook is built around the local MultiBypass140 label manifests mirrored in this repository. It focuses on the questions that matter for BariatricRSD:

- How many videos, frames, centers, splits, phases, and phase-order clusters are present?
- What does the remaining surgery duration (RSD) label look like for every video?
- How different are Bern and Strasbourg in duration, phase mix, deviation burden, and phase order?
- Which videos should be inspected visually?
- What local training runs and checkpoints already exist?

Raw frames/videos are not committed to this repository. Media cells automatically render examples when `DATA_ROOT` points at a mounted MultiBypass140 data directory.


## 0. Configuration

Run this notebook from the repository root or from the `notebooks/` directory. If the raw MultiBypass140 media is mounted, set `DATA_ROOT` to the directory containing `BernBypass70/` and `StrasBypass70/`.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import os
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

try:
    from IPython.display import display, Markdown, Video
except Exception:
    def display(x):
        print(x)
    def Markdown(x):
        return x
    Video = None

try:
    from PIL import Image as PILImage
except Exception:
    PILImage = None

try:
    import cv2
except Exception:
    cv2 = None

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

plt.rcParams.update({
    'figure.figsize': (11, 5),
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent

LABEL_DIR = PROJECT / 'lambda_mirror' / 'labels'
OUTPUT_DIR = PROJECT / 'lambda_mirror' / 'outputs'
LOG_DIR = PROJECT / 'lambda_mirror' / 'logs'
WANDB_DIR = PROJECT / 'lambda_mirror' / 'wandb'
FIG_DIR = PROJECT / 'notebooks' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT_CANDIDATES = [
    PROJECT / 'extern' / 'MultiBypass140' / 'datasets' / 'MultiBypass140',
    PROJECT / 'lambda_mirror' / 'extern' / 'MultiBypass140' / 'datasets' / 'MultiBypass140',
    Path('/lambda/nfs/bariatric-rsd/extern/MultiBypass140/datasets/MultiBypass140'),
    Path('/home/ubuntu/bariatric-rsd/extern/MultiBypass140/datasets/MultiBypass140'),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), None)

LABEL_PATH = LABEL_DIR / 'mb140_fold0_labels_kmeans.json'
if not LABEL_PATH.exists():
    label_candidates = sorted(LABEL_DIR.glob('*.json'))
    if not label_candidates:
        raise FileNotFoundError(f'No label JSONs found under {LABEL_DIR}')
    LABEL_PATH = label_candidates[0]

CENTER_COLORS = {'Bern': '#2563eb', 'Strasbourg': '#dc2626', 'Unknown': '#525252'}
SPLIT_COLORS = {'train': '#16a34a', 'val': '#f59e0b', 'test': '#7c3aed', 'missing': '#737373'}

print(f'PROJECT: {PROJECT}')
print(f'LABEL_PATH: {LABEL_PATH.relative_to(PROJECT)}')
print(f'DATA_ROOT: {DATA_ROOT if DATA_ROOT else "not found; media cells will explain how to enable previews"}')
print(f'FIG_DIR: {FIG_DIR.relative_to(PROJECT)}')


## 1. Manifest Inventory

The project has multiple label manifests. The default used below is the fold-0 manifest with k-means phase-order clusters: `mb140_fold0_labels_kmeans.json`.


In [ ]:
def center_from_video_id(video_id: str) -> str:
    video_id = str(video_id)
    if video_id.startswith('BBP'):
        return 'Bern'
    if video_id.startswith('SBP'):
        return 'Strasbourg'
    return 'Unknown'


def hhmmss(seconds: float) -> str:
    seconds = int(round(float(seconds)))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2)

manifest_rows = []
for path in sorted(LABEL_DIR.glob('*.json')):
    with open(path, 'r') as f:
        manifest = json.load(f)
    duration_min = [v.get('total_duration_sec', np.nan) / 60 for v in manifest]
    manifest_rows.append({
        'manifest': path.name,
        'size_mb': size_mb(path),
        'videos': len(manifest),
        'frames': sum(len(v.get('frames', [])) for v in manifest),
        'median_duration_min': np.nanmedian(duration_min),
        'mean_duration_min': np.nanmean(duration_min),
        'splits': dict(Counter(v.get('split', 'missing') for v in manifest)),
        'clusters': dict(Counter(v.get('phase_order_cluster', 'missing') for v in manifest)),
    })

manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df)


## 2. Load Dataframes

The notebook creates two tables:

- `video_df`: one row per surgery video.
- `frame_df`: one row per labeled frame/sample, including timestamp, RSD, phase, and deviation label.

`frame_df` has about 771k rows, which is small enough for local Pandas analysis.


In [ ]:
with open(LABEL_PATH, 'r') as f:
    videos = json.load(f)

print(f'Loaded {len(videos):,} videos')
print('Video keys:', list(videos[0].keys()))
print('Frame keys:', list(videos[0]['frames'][0].keys()))


def compressed_phase_sequence(frames):
    seq = []
    previous = None
    for frame in frames:
        phase = frame.get('phase', 'Unknown')
        if phase != previous:
            seq.append(phase)
            previous = phase
    return seq

video_rows = []
frame_rows = []
segment_rows = []
event_rows = []

for v in videos:
    video_id = v['video_id']
    center = center_from_video_id(video_id)
    split = v.get('split', 'missing')
    cluster = v.get('phase_order_cluster')
    frames = v.get('frames', [])
    total_sec = float(v.get('total_duration_sec', np.nan))
    first_ts = float(frames[0].get('timestamp_sec', np.nan)) if frames else np.nan
    last_ts = float(frames[-1].get('timestamp_sec', np.nan)) if frames else np.nan
    seq = v.get('phase_sequence') or compressed_phase_sequence(frames)
    phase_counts = Counter(frame.get('phase', 'Unknown') for frame in frames)
    n_dev_frames = sum(1 for frame in frames if frame.get('is_deviation'))

    video_rows.append({
        'video_id': video_id,
        'center': center,
        'split': split,
        'phase_order_cluster': cluster,
        'duration_sec': total_sec,
        'duration_min': total_sec / 60,
        'n_frames': len(frames),
        'first_timestamp_sec': first_ts,
        'last_timestamp_sec': last_ts,
        'labeled_span_sec': last_ts - first_ts + 1 if frames else np.nan,
        'n_unique_phases': len(phase_counts),
        'n_phase_segments': len(compressed_phase_sequence(frames)),
        'n_deviation_frames': n_dev_frames,
        'deviation_frame_pct': n_dev_frames / len(frames) if frames else np.nan,
        'phase_sequence': ' -> '.join(seq),
        'phase_sequence_len': len(seq),
    })

    previous_phase = None
    segment = None
    active_event = None

    for frame in frames:
        ts = float(frame.get('timestamp_sec', np.nan))
        phase = frame.get('phase', 'Unknown')
        is_dev = bool(frame.get('is_deviation'))
        elapsed_sec = ts - first_ts if pd.notna(first_ts) else np.nan
        rsd_sec = float(frame.get('rsd_sec', np.nan))

        frame_rows.append({
            'video_id': video_id,
            'center': center,
            'split': split,
            'phase_order_cluster': cluster,
            'frame_idx': frame.get('frame_idx'),
            'timestamp_sec': ts,
            'elapsed_sec': elapsed_sec,
            'timestamp_min': ts / 60,
            'elapsed_min': elapsed_sec / 60,
            'duration_min': total_sec / 60,
            'progress': ts / total_sec if total_sec and pd.notna(ts) else np.nan,
            'elapsed_progress': elapsed_sec / max(1, (last_ts - first_ts)) if pd.notna(elapsed_sec) and pd.notna(last_ts) and pd.notna(first_ts) else np.nan,
            'rsd_sec': rsd_sec,
            'rsd_min': rsd_sec / 60,
            'rsd_normalized': frame.get('rsd_normalized'),
            'phase': phase,
            'is_deviation': is_dev,
            'frame_path': frame.get('frame_path'),
        })

        if phase != previous_phase:
            if segment is not None:
                segment['end_sec'] = previous_ts
                segment['end_min'] = previous_ts / 60
                segment['duration_sec'] = max(0, previous_ts - segment['start_sec'] + 1)
                segment_rows.append(segment)
            segment = {
                'video_id': video_id,
                'center': center,
                'split': split,
                'phase_order_cluster': cluster,
                'phase': phase,
                'start_sec': ts,
                'start_min': ts / 60,
            }
            previous_phase = phase

        if is_dev and active_event is None:
            active_event = {
                'video_id': video_id,
                'center': center,
                'split': split,
                'phase_order_cluster': cluster,
                'start_sec': ts,
                'end_sec': ts,
                'dominant_phase_counts': Counter([phase]),
                'frames': 1,
            }
        elif is_dev and active_event is not None:
            active_event['end_sec'] = ts
            active_event['dominant_phase_counts'].update([phase])
            active_event['frames'] += 1
        elif (not is_dev) and active_event is not None:
            active_event['duration_sec'] = max(0, active_event['end_sec'] - active_event['start_sec'] + 1)
            active_event['dominant_phase'] = active_event['dominant_phase_counts'].most_common(1)[0][0]
            active_event.pop('dominant_phase_counts')
            event_rows.append(active_event)
            active_event = None

        previous_ts = ts

    if segment is not None:
        segment['end_sec'] = previous_ts
        segment['end_min'] = previous_ts / 60
        segment['duration_sec'] = max(0, previous_ts - segment['start_sec'] + 1)
        segment_rows.append(segment)

    if active_event is not None:
        active_event['duration_sec'] = max(0, active_event['end_sec'] - active_event['start_sec'] + 1)
        active_event['dominant_phase'] = active_event['dominant_phase_counts'].most_common(1)[0][0]
        active_event.pop('dominant_phase_counts')
        event_rows.append(active_event)

video_df = pd.DataFrame(video_rows).sort_values('video_id').reset_index(drop=True)
frame_df = pd.DataFrame(frame_rows)
segment_df = pd.DataFrame(segment_rows)
event_df = pd.DataFrame(event_rows)

for col in ['video_id', 'center', 'split', 'phase', 'phase_order_cluster']:
    if col in frame_df:
        frame_df[col] = frame_df[col].astype('category')
for col in ['video_id', 'center', 'split', 'phase', 'phase_order_cluster']:
    if col in segment_df:
        segment_df[col] = segment_df[col].astype('category')

print('video_df:', video_df.shape)
print('frame_df:', frame_df.shape)
print('segment_df:', segment_df.shape)
print('event_df:', event_df.shape)
display(video_df.head())


## 3. Executive Summary

These are the headline numbers for the selected manifest.


In [ ]:
summary = pd.DataFrame([
    {'metric': 'videos', 'value': len(video_df)},
    {'metric': 'frames / samples', 'value': int(video_df['n_frames'].sum())},
    {'metric': 'centers', 'value': video_df['center'].nunique()},
    {'metric': 'splits', 'value': ', '.join(sorted(video_df['split'].unique()))},
    {'metric': 'phase-order clusters', 'value': video_df['phase_order_cluster'].nunique()},
    {'metric': 'median duration min', 'value': round(video_df['duration_min'].median(), 2)},
    {'metric': 'mean duration min', 'value': round(video_df['duration_min'].mean(), 2)},
    {'metric': 'min duration min', 'value': round(video_df['duration_min'].min(), 2)},
    {'metric': 'max duration min', 'value': round(video_df['duration_min'].max(), 2)},
    {'metric': 'deviation frames', 'value': int(video_df['n_deviation_frames'].sum())},
    {'metric': 'deviation frame rate', 'value': f"{video_df['n_deviation_frames'].sum() / video_df['n_frames'].sum():.2%}"},
    {'metric': 'deviation events', 'value': len(event_df)},
])
display(summary)

display(video_df.groupby('center').agg(
    videos=('video_id', 'count'),
    frames=('n_frames', 'sum'),
    median_duration_min=('duration_min', 'median'),
    mean_duration_min=('duration_min', 'mean'),
    max_duration_min=('duration_min', 'max'),
    median_deviation_pct=('deviation_frame_pct', 'median'),
    videos_with_deviation=('n_deviation_frames', lambda s: int((s > 0).sum())),
).round(3))

display(pd.crosstab(video_df['split'], video_df['center'], margins=True))


## 4. RSD for Every Video: Linear Plot

This is the plot you asked for. Each line is one surgery video. The x-axis is elapsed operative time in minutes and the y-axis is remaining surgery duration in minutes. Since RSD is defined as remaining time, every line should be approximately linear with slope -1.

The line height at x = 0 is the total labeled duration. Longer surgeries start higher and reach zero later.


In [ ]:
def plot_all_video_rsd_lines(video_table=video_df, save_path=None):
    fig, ax = plt.subplots(figsize=(13, 8))
    ordered = video_table.sort_values(['center', 'duration_min'])
    for row in ordered.itertuples(index=False):
        color = CENTER_COLORS.get(row.center, CENTER_COLORS['Unknown'])
        ax.plot(
            [0, row.duration_min],
            [row.duration_min, 0],
            color=color,
            alpha=0.42,
            linewidth=1.15,
        )
    for center, group in video_table.groupby('center'):
        median_duration = group['duration_min'].median()
        ax.plot(
            [0, median_duration],
            [median_duration, 0],
            color=CENTER_COLORS.get(center, CENTER_COLORS['Unknown']),
            linewidth=3.2,
            alpha=0.95,
            label=f'{center} median ({median_duration:.1f} min)',
        )
    ax.set_title('Remaining Surgery Duration Labels for Every Video')
    ax.set_xlabel('Elapsed operative time (min)')
    ax.set_ylabel('Remaining surgery duration, RSD (min)')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.legend(loc='upper right')
    ax.text(
        0.01,
        0.02,
        f'n={len(video_table)} videos; each thin line is one video',
        transform=ax.transAxes,
        ha='left',
        va='bottom',
        color='#404040',
    )
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=220, bbox_inches='tight')
    return fig, ax

plot_all_video_rsd_lines(save_path=FIG_DIR / 'rsd_all_videos_linear.png')
plt.show()


Static preview generated by the previous cell:

![All per-video linear RSD curves](figures/rsd_all_videos_linear.png)


In [ ]:
def plot_rsd_lines_by_split(video_table=video_df, save_path=None):
    splits = [s for s in ['train', 'val', 'test'] if s in set(video_table['split'])]
    fig, axes = plt.subplots(1, len(splits), figsize=(5.2 * len(splits), 5.3), sharex=True, sharey=True)
    if len(splits) == 1:
        axes = [axes]
    for ax, split in zip(axes, splits):
        subset = video_table[video_table['split'] == split].sort_values('duration_min')
        for row in subset.itertuples(index=False):
            ax.plot(
                [0, row.duration_min],
                [row.duration_min, 0],
                color=CENTER_COLORS.get(row.center, CENTER_COLORS['Unknown']),
                alpha=0.45,
                linewidth=1.2,
            )
        ax.set_title(f'{split}: {len(subset)} videos')
        ax.set_xlabel('Elapsed time (min)')
        ax.set_ylabel('RSD (min)')
    legend = [Line2D([0], [0], color=color, lw=3, label=center) for center, color in CENTER_COLORS.items() if center in set(video_table['center'])]
    axes[-1].legend(handles=legend, loc='upper right')
    fig.suptitle('Per-Video RSD Lines by Split', y=1.02)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=220, bbox_inches='tight')
    return fig, axes

plot_rsd_lines_by_split(save_path=FIG_DIR / 'rsd_lines_by_split.png')
plt.show()


Static preview:

![Per-video RSD curves by split](figures/rsd_lines_by_split.png)


In [ ]:
def plot_actual_frame_rsd_overlay(max_points_per_video=220, save_path=None):
    fig, ax = plt.subplots(figsize=(12, 6.5))
    for video_id, group in frame_df.groupby('video_id', observed=True):
        group = group.sort_values('elapsed_min')
        if len(group) > max_points_per_video:
            group = group.iloc[np.linspace(0, len(group) - 1, max_points_per_video).astype(int)]
        center = str(group['center'].iloc[0])
        ax.plot(
            group['elapsed_min'],
            group['rsd_min'],
            color=CENTER_COLORS.get(center, CENTER_COLORS['Unknown']),
            alpha=0.18,
            linewidth=0.9,
        )
    ax.set_title('Actual Frame-Level RSD Labels: All Videos')
    ax.set_xlabel('Elapsed time since first labeled frame (min)')
    ax.set_ylabel('RSD label (min)')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.legend(handles=[Line2D([0], [0], color=c, lw=3, label=k) for k, c in CENTER_COLORS.items() if k in set(video_df['center'])])
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=220, bbox_inches='tight')
    return fig, ax

plot_actual_frame_rsd_overlay(save_path=FIG_DIR / 'rsd_actual_frame_overlay.png')
plt.show()


In [ ]:
def plot_normalized_rsd_sanity(save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    sampled_groups = []
    for _, group in frame_df.groupby('video_id', observed=True):
        group = group.sort_values('elapsed_min')
        idx = np.linspace(0, len(group) - 1, min(80, len(group))).astype(int)
        sampled_groups.append(group.iloc[idx])
    sample = pd.concat(sampled_groups, ignore_index=True)

    for video_id, group in sample.groupby('video_id', observed=True):
        center = str(group['center'].iloc[0])
        axes[0].plot(group['progress'], group['rsd_normalized'], color=CENTER_COLORS.get(center), alpha=0.15, linewidth=0.8)
        axes[1].plot(group['elapsed_progress'], group['rsd_min'] / group['duration_min'], color=CENTER_COLORS.get(center), alpha=0.15, linewidth=0.8)

    axes[0].set_title('Manifest rsd_normalized vs timestamp / total duration')
    axes[0].set_xlabel('timestamp_sec / total_duration_sec')
    axes[0].set_ylabel('rsd_normalized')
    axes[1].set_title('RSD / duration vs elapsed labeled progress')
    axes[1].set_xlabel('elapsed progress through labeled frames')
    axes[1].set_ylabel('rsd_min / duration_min')
    for ax in axes:
        ax.set_xlim(0, 1.02)
        ax.set_ylim(-0.02, 1.02)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=220, bbox_inches='tight')
    return fig, axes

plot_normalized_rsd_sanity(save_path=FIG_DIR / 'rsd_normalized_sanity.png')
plt.show()


Interpretation: the linear RSD plot is intentionally simple, but it is diagnostically useful. If any curve bends, jumps, or fails to reach zero, the RSD label construction is wrong. The main variation here is the intercept, which is total procedure duration.


In [ ]:
rsd_video_table = video_df[[
    'video_id', 'center', 'split', 'phase_order_cluster', 'duration_min', 'n_frames',
    'first_timestamp_sec', 'last_timestamp_sec', 'n_deviation_frames', 'deviation_frame_pct',
]].copy()
rsd_video_table['duration_hhmmss'] = rsd_video_table['duration_min'].mul(60).map(hhmmss)
rsd_video_table = rsd_video_table.sort_values('duration_min', ascending=False)

display(rsd_video_table.head(20))

display(rsd_video_table.groupby(['center', 'split']).agg(
    videos=('video_id', 'count'),
    median_duration_min=('duration_min', 'median'),
    mean_duration_min=('duration_min', 'mean'),
    longest_duration_min=('duration_min', 'max'),
).round(2))


## 5. Individual Video RSD Timelines

The next function plots one video with:

- RSD as a black descending line.
- Phase spans as colored background blocks.
- Deviation frames as red markers on top of the RSD line.

This is the most useful plot for inspecting whether deviations and phase transitions happen early or late in a procedure.


In [ ]:
PHASES = list(frame_df['phase'].cat.categories) if hasattr(frame_df['phase'], 'cat') else sorted(frame_df['phase'].unique())
PHASE_COLORS = {phase: plt.cm.tab20(i % 20) for i, phase in enumerate(PHASES)}


def plot_video_rsd_timeline(video_id, save_path=None):
    group = frame_df[frame_df['video_id'].astype(str) == str(video_id)].sort_values('elapsed_min')
    if group.empty:
        raise ValueError(f'Unknown video_id: {video_id}')
    segments = segment_df[segment_df['video_id'].astype(str) == str(video_id)].copy()
    first_ts = group['timestamp_sec'].min()
    segments['elapsed_start_min'] = (segments['start_sec'] - first_ts) / 60
    segments['elapsed_end_min'] = (segments['end_sec'] - first_ts) / 60

    fig, ax = plt.subplots(figsize=(14, 5.8))
    ymax = group['rsd_min'].max()
    for seg in segments.itertuples(index=False):
        ax.axvspan(
            seg.elapsed_start_min,
            seg.elapsed_end_min,
            color=PHASE_COLORS.get(str(seg.phase), '#e5e5e5'),
            alpha=0.18,
            linewidth=0,
        )
    ax.plot(group['elapsed_min'], group['rsd_min'], color='#111827', linewidth=2.2, label='RSD')

    dev = group[group['is_deviation']]
    if not dev.empty:
        ax.scatter(dev['elapsed_min'], dev['rsd_min'], s=13, color='#dc2626', alpha=0.8, label='Deviation frame')

    meta = video_df[video_df['video_id'] == str(video_id)].iloc[0]
    ax.set_title(
        f"{video_id}: RSD timeline | {meta['center']} | {meta['split']} | "
        f"cluster {meta['phase_order_cluster']} | duration {meta['duration_min']:.1f} min"
    )
    ax.set_xlabel('Elapsed time since first labeled frame (min)')
    ax.set_ylabel('Remaining surgery duration (min)')
    ax.set_ylim(0, ymax * 1.06)
    ax.set_xlim(0, group['elapsed_min'].max())

    phase_handles = [Patch(facecolor=PHASE_COLORS[p], alpha=0.25, label=p) for p in segments['phase'].astype(str).drop_duplicates().head(12)]
    line_handles = [Line2D([0], [0], color='#111827', lw=2.2, label='RSD')]
    if not dev.empty:
        line_handles.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='#dc2626', markersize=7, label='Deviation frame'))
    ax.legend(handles=line_handles + phase_handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=220, bbox_inches='tight')
    return fig, ax

example_video_ids = [
    video_df.sort_values('duration_min').iloc[0]['video_id'],
    video_df.iloc[(video_df['duration_min'] - video_df['duration_min'].median()).abs().argsort().iloc[0]]['video_id'],
    video_df.sort_values('duration_min', ascending=False).iloc[0]['video_id'],
    video_df.sort_values('deviation_frame_pct', ascending=False).iloc[0]['video_id'],
]
example_video_ids = list(dict.fromkeys(example_video_ids))
example_video_ids


In [ ]:
for vid in example_video_ids:
    plot_video_rsd_timeline(vid, save_path=FIG_DIR / f'rsd_timeline_{vid}.png')
    plt.show()


## 6. Duration, Center, Split, and Cluster Structure

The RSD labels are linear, so the modeling challenge is not the shape of the label; it is estimating where the current case sits in a highly variable procedure duration distribution.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for center, group in video_df.groupby('center'):
    axes[0, 0].hist(group['duration_min'], bins=18, alpha=0.62, color=CENTER_COLORS.get(center), label=center)
axes[0, 0].set_title('Duration Distribution by Center')
axes[0, 0].set_xlabel('Duration (min)')
axes[0, 0].set_ylabel('Videos')
axes[0, 0].legend()

pd.crosstab(video_df['split'], video_df['center']).plot(kind='bar', ax=axes[0, 1], rot=0, color=[CENTER_COLORS.get(c) for c in sorted(video_df['center'].unique())])
axes[0, 1].set_title('Split Balance by Center')
axes[0, 1].set_xlabel('Split')
axes[0, 1].set_ylabel('Videos')

video_df.boxplot(column='duration_min', by='phase_order_cluster', ax=axes[1, 0])
axes[1, 0].set_title('Duration by Phase-Order Cluster')
axes[1, 0].set_xlabel('Cluster')
axes[1, 0].set_ylabel('Duration (min)')

axes[1, 1].scatter(
    video_df['duration_min'],
    video_df['deviation_frame_pct'] * 100,
    c=video_df['center'].map(CENTER_COLORS),
    alpha=0.82,
    s=42,
)
axes[1, 1].set_title('Deviation Burden vs Duration')
axes[1, 1].set_xlabel('Duration (min)')
axes[1, 1].set_ylabel('Deviation frames (%)')
axes[1, 1].legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor=color, label=center, markersize=8) for center, color in CENTER_COLORS.items() if center in set(video_df['center'])])

plt.suptitle('')
fig.tight_layout()
fig.savefig(FIG_DIR / 'dataset_structure.png', dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
cluster_summary = video_df.groupby('phase_order_cluster').agg(
    videos=('video_id', 'count'),
    bern=('center', lambda s: int((s == 'Bern').sum())),
    strasbourg=('center', lambda s: int((s == 'Strasbourg').sum())),
    median_duration_min=('duration_min', 'median'),
    mean_duration_min=('duration_min', 'mean'),
    median_deviation_pct=('deviation_frame_pct', 'median'),
    mean_phase_segments=('n_phase_segments', 'mean'),
).sort_values('videos', ascending=False).round(3)

display(cluster_summary)

ax = pd.crosstab(video_df['phase_order_cluster'], video_df['center']).plot(kind='bar', figsize=(10, 4.5), rot=0)
ax.set_title('Phase-Order Cluster Composition by Center')
ax.set_xlabel('Phase-order cluster')
ax.set_ylabel('Videos')
plt.tight_layout()
plt.show()


## 7. Phase Statistics

Frame counts are approximately seconds because the manifest is sampled at about 1 fps. Phase minutes below are therefore approximate labeled minutes.


In [ ]:
phase_stats = frame_df.groupby('phase', observed=True).agg(
    frames=('phase', 'size'),
    videos_present=('video_id', lambda s: s.nunique()),
    deviation_frames=('is_deviation', 'sum'),
).reset_index()
phase_stats['minutes_at_1fps'] = phase_stats['frames'] / 60
phase_stats['frame_pct'] = phase_stats['frames'] / phase_stats['frames'].sum()
phase_stats['deviation_rate_within_phase'] = phase_stats['deviation_frames'] / phase_stats['frames']
phase_stats['share_of_deviation_frames'] = phase_stats['deviation_frames'] / max(1, phase_stats['deviation_frames'].sum())
phase_stats = phase_stats.sort_values('frames', ascending=False).reset_index(drop=True)

display(phase_stats)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
phase_stats.sort_values('minutes_at_1fps').plot.barh(x='phase', y='minutes_at_1fps', ax=axes[0], legend=False, color='#2563eb')
axes[0].set_title('Frame-Weighted Phase Duration')
axes[0].set_xlabel('Approx labeled minutes')
axes[0].set_ylabel('')

phase_stats.sort_values('deviation_rate_within_phase').plot.barh(x='phase', y='deviation_rate_within_phase', ax=axes[1], legend=False, color='#dc2626')
axes[1].set_title('Deviation Rate Within Phase')
axes[1].set_xlabel('Deviation frames / phase frames')
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter(lambda x, pos: f'{100*x:.0f}%')

fig.tight_layout()
fig.savefig(FIG_DIR / 'phase_statistics.png', dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
phase_center = pd.crosstab(frame_df['phase'], frame_df['center'], normalize='columns')
phase_center = phase_center.loc[phase_stats['phase']]
display(phase_center.style.format('{:.1%}'))

fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(phase_center.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(phase_center.columns)), phase_center.columns)
ax.set_yticks(range(len(phase_center.index)), phase_center.index)
ax.set_title('Phase Mix by Center')
for i in range(phase_center.shape[0]):
    for j in range(phase_center.shape[1]):
        ax.text(j, i, f'{phase_center.iloc[i, j]:.0%}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()


## 8. Phase Transitions and Orderings

This section is for evaluating the phase-order token idea. It shows which phase transitions are common and which ordered workflows dominate.


In [ ]:
transition_counts = defaultdict(Counter)
sequence_counts = Counter()
for v in videos:
    seq = compressed_phase_sequence(v.get('frames', []))
    sequence_counts[' -> '.join(seq)] += 1
    for src, dst in zip(seq, seq[1:]):
        transition_counts[src][dst] += 1

ordered_phases = phase_stats['phase'].tolist()
transition_df = pd.DataFrame(0, index=ordered_phases, columns=ordered_phases, dtype=int)
for src, dests in transition_counts.items():
    for dst, n in dests.items():
        if src in transition_df.index and dst in transition_df.columns:
            transition_df.loc[src, dst] = n

row_norm = transition_df.div(transition_df.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

display(pd.DataFrame({'n_videos': sequence_counts}).reset_index(names='phase_sequence').sort_values('n_videos', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(row_norm.values, cmap='magma', aspect='auto')
ax.set_xticks(range(len(ordered_phases)), ordered_phases, rotation=75, ha='right')
ax.set_yticks(range(len(ordered_phases)), ordered_phases)
ax.set_title('Row-Normalized Phase Transition Matrix')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / 'phase_transition_matrix.png', dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
for cluster, group in video_df.groupby('phase_order_cluster'):
    print(f'Cluster {cluster}: {len(group)} videos | centers={dict(Counter(group["center"]))}')
    display(group[['video_id', 'center', 'split', 'duration_min', 'n_phase_segments', 'phase_sequence']].head(6))


## 9. Deviation / IAE Analysis

The local manifest exposes a binary `is_deviation` frame label. Category-level IAE analysis requires the raw `with_iae` pickle labels from the official dataset.


In [ ]:
if event_df.empty:
    print('No deviation events found in this manifest.')
else:
    event_df = event_df.copy()
    event_df['duration_min'] = event_df['duration_sec'] / 60
    display(event_df.sort_values('duration_sec', ascending=False).head(20))
    display(event_df.groupby(['center', 'split']).agg(
        events=('video_id', 'count'),
        videos=('video_id', 'nunique'),
        median_event_sec=('duration_sec', 'median'),
        total_event_min=('duration_min', 'sum'),
    ).round(2))


In [ ]:
if not event_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    event_df['duration_sec'].clip(upper=600).hist(bins=30, ax=axes[0], color='#dc2626')
    axes[0].set_title('Deviation Event Duration, Clipped at 10 min')
    axes[0].set_xlabel('Duration (sec)')
    axes[0].set_ylabel('Events')

    event_df['dominant_phase'].value_counts().sort_values().plot.barh(ax=axes[1], color='#dc2626')
    axes[1].set_title('Deviation Events by Dominant Phase')
    axes[1].set_xlabel('Events')

    video_df.assign(has_deviation=video_df['n_deviation_frames'] > 0).groupby(['center', 'has_deviation']).size().unstack(fill_value=0).plot(kind='bar', ax=axes[2], rot=0)
    axes[2].set_title('Videos With Any Deviation')
    axes[2].set_xlabel('Center')
    axes[2].set_ylabel('Videos')

    fig.tight_layout()
    fig.savefig(FIG_DIR / 'deviation_statistics.png', dpi=220, bbox_inches='tight')
    plt.show()


## 10. Media Preview Helpers

These cells render real images and videos only if the raw data is mounted. The manifest frame paths are relative paths such as `BernBypass70/frames/BBP01/BBP01_00000014.jpg`.


In [ ]:
def resolve_frame_path(relative_path: str):
    rel = Path(str(relative_path))
    candidates = []
    if DATA_ROOT is not None:
        candidates.append(DATA_ROOT / rel)
    candidates.extend([PROJECT / rel, PROJECT / 'lambda_mirror' / rel])
    for path in candidates:
        if path.exists():
            return path
    return None

sample_paths = []
for _, row in frame_df.groupby('video_id', observed=True).head(1).iterrows():
    resolved = resolve_frame_path(row['frame_path'])
    sample_paths.append({
        'video_id': str(row['video_id']),
        'phase': str(row['phase']),
        'manifest_path': row['frame_path'],
        'exists': resolved is not None,
        'resolved_path': str(resolved) if resolved else None,
    })
media_check = pd.DataFrame(sample_paths)
display(media_check.head(20))
print(f'Sampled frame path existence: {media_check["exists"].mean():.1%}')


In [ ]:
def sample_existing_frames(per_phase=1, max_total=16):
    chosen = []
    seen = Counter()
    for row in frame_df.sort_values(['phase', 'video_id', 'elapsed_min']).itertuples(index=False):
        phase = str(row.phase)
        if seen[phase] >= per_phase:
            continue
        path = resolve_frame_path(row.frame_path)
        if path is not None:
            chosen.append({
                'path': path,
                'video_id': str(row.video_id),
                'phase': phase,
                'elapsed_min': row.elapsed_min,
                'is_deviation': row.is_deviation,
            })
            seen[phase] += 1
        if len(chosen) >= max_total:
            break
    return chosen


def show_image_grid(records, cols=4):
    if PILImage is None:
        display(Markdown('PIL is not available.'))
        return
    if not records:
        display(Markdown('No local frame files were found. Set `DATA_ROOT` to the MultiBypass140 media root and rerun.'))
        return
    rows = math.ceil(len(records) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0))
    axes = np.array(axes).reshape(-1)
    for ax, rec in zip(axes, records):
        img = PILImage.open(rec['path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"{rec['video_id']} | {rec['phase']}\n{rec['elapsed_min']:.1f} min", fontsize=8)
        ax.axis('off')
    for ax in axes[len(records):]:
        ax.axis('off')
    fig.tight_layout()
    plt.show()

show_image_grid(sample_existing_frames(per_phase=1, max_total=16), cols=4)


In [ ]:
def center_folder(video_id: str) -> str:
    return 'BernBypass70' if str(video_id).startswith('BBP') else 'StrasBypass70'


def resolve_raw_video_path(video_id: str):
    if DATA_ROOT is None:
        return None
    base = DATA_ROOT / center_folder(video_id) / 'videos'
    for suffix in ['mp4', 'avi', 'mov', 'mkv', 'webm']:
        for name in [video_id, str(video_id).lower(), str(video_id).upper()]:
            path = base / f'{name}.{suffix}'
            if path.exists():
                return path
    return None


def show_raw_video(video_id=None, width=760):
    if video_id is None:
        video_id = video_df.iloc[0]['video_id']
    path = resolve_raw_video_path(video_id)
    if path is None:
        expected = DATA_ROOT / center_folder(video_id) / 'videos' if DATA_ROOT else 'DATA_ROOT/<Center>/videos'
        display(Markdown(f'No raw video found for `{video_id}`. Expected under `{expected}`.'))
        return None
    display(Markdown(f'Raw video: `{path}`'))
    if Video is not None:
        display(Video(str(path), embed=False, width=width))
    return path

show_raw_video(video_df.iloc[0]['video_id'])


## 11. Training Runs and Checkpoints

This parses mirrored Lambda logs and W&B summaries without loading checkpoint tensors.


In [ ]:
VAL_RE = re.compile(r'Epoch\s+(\d+)\s+\|\s+val_mae=([0-9.]+)min\s+\|\s+pearson_r=([\-0-9.]+)\s+\|\s+dev_f1=([\-0-9.]+)\s+\|\s+best=([0-9.]+)min')
DATASET_RE = re.compile(r'BariatricFrameDataset \[(train|val|test)\]:\s+(\d+) videos,\s+(\d+) samples')
PARAM_RE = re.compile(r'Total params:\s+([0-9.]+)M\s+\|\s+Trainable:\s+([0-9.]+)M')
WANDB_RUN_RE = re.compile(r'Syncing run\s+([^\s]+)')

log_paths = sorted(LOG_DIR.glob('*.log')) if LOG_DIR.exists() else []
metric_rows = []
run_rows = []
for path in log_paths:
    text = path.read_text(errors='replace')
    run_match = WANDB_RUN_RE.search(text)
    run_name = run_match.group(1) if run_match else path.stem
    datasets = {m.group(1): {'videos': int(m.group(2)), 'samples': int(m.group(3))} for m in DATASET_RE.finditer(text)}
    params = PARAM_RE.search(text)
    vals = []
    for m in VAL_RE.finditer(text):
        row = {
            'run': run_name,
            'log_file': str(path.relative_to(PROJECT)),
            'epoch': int(m.group(1)),
            'val_mae_min': float(m.group(2)),
            'pearson_r': float(m.group(3)),
            'dev_f1': float(m.group(4)),
            'best_mae_min': float(m.group(5)),
        }
        vals.append(row)
        metric_rows.append(row)
    run_rows.append({
        'run': run_name,
        'log_file': str(path.relative_to(PROJECT)),
        'epochs_logged': len(vals),
        'best_val_mae_min': min([r['val_mae_min'] for r in vals], default=np.nan),
        'best_dev_f1': max([r['dev_f1'] for r in vals], default=np.nan),
        'final_val_mae_min': vals[-1]['val_mae_min'] if vals else np.nan,
        'train_videos': datasets.get('train', {}).get('videos'),
        'train_samples': datasets.get('train', {}).get('samples'),
        'val_videos': datasets.get('val', {}).get('videos'),
        'val_samples': datasets.get('val', {}).get('samples'),
        'total_params_m': float(params.group(1)) if params else np.nan,
        'trainable_params_m': float(params.group(2)) if params else np.nan,
    })

runs_df = pd.DataFrame(run_rows).sort_values('best_val_mae_min', na_position='last')
metrics_df = pd.DataFrame(metric_rows)
display(runs_df)


In [ ]:
if not metrics_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for run, group in metrics_df.groupby('run'):
        axes[0].plot(group['epoch'], group['val_mae_min'], marker='o', linewidth=1.5, label=run)
        axes[1].plot(group['epoch'], group['dev_f1'], marker='o', linewidth=1.5, label=run)
    axes[0].set_title('Validation RSD MAE')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MAE (min)')
    axes[1].set_title('Deviation F1')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1')
    axes[0].legend(fontsize=7)
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'training_run_metrics.png', dpi=220, bbox_inches='tight')
    plt.show()
else:
    print('No validation metrics parsed from logs.')


In [ ]:
ckpt_rows = []
for path in sorted(OUTPUT_DIR.glob('**/*.pth')) if OUTPUT_DIR.exists() else []:
    ckpt_rows.append({
        'path': str(path.relative_to(PROJECT)),
        'run_dir': path.parent.name,
        'filename': path.name,
        'size_gb': path.stat().st_size / (1024 ** 3),
        'modified': pd.Timestamp.fromtimestamp(path.stat().st_mtime),
    })
ckpt_df = pd.DataFrame(ckpt_rows)
display(ckpt_df.sort_values(['run_dir', 'filename']) if not ckpt_df.empty else ckpt_df)

display(ckpt_df.groupby('run_dir').agg(checkpoints=('filename', 'count'), total_gb=('size_gb', 'sum')).sort_values('total_gb', ascending=False) if not ckpt_df.empty else pd.DataFrame())


## 12. Data Quality Checks

These checks are intentionally practical. Anything marked `review` is not necessarily wrong, but it is something to understand before reporting model results.


In [ ]:
checks = []

def add_check(name, status, detail):
    checks.append({'check': name, 'status': status, 'detail': detail})

add_check('duplicate video IDs', 'pass' if video_df['video_id'].is_unique else 'fail', f"duplicates={video_df['video_id'].duplicated().sum()}")
add_check('empty videos', 'pass' if (video_df['n_frames'] > 0).all() else 'fail', f"empty={(video_df['n_frames'] == 0).sum()}")
add_check('center balance', 'pass' if set(video_df['center'].value_counts()) == {70} else 'review', str(dict(video_df['center'].value_counts())))
add_check('split balance', 'pass' if set(video_df['split']) >= {'train', 'val'} else 'review', str(dict(video_df['split'].value_counts())))
add_check('cluster labels present', 'pass' if video_df['phase_order_cluster'].notna().all() else 'fail', f"missing={video_df['phase_order_cluster'].isna().sum()}")
add_check('duration reaches zero RSD', 'pass' if frame_df.groupby('video_id', observed=True)['rsd_sec'].min().max() <= 1 else 'review', f"max video min_rsd_sec={frame_df.groupby('video_id', observed=True)['rsd_sec'].min().max():.2f}")
add_check('RSD monotone non-increasing', 'pass' if frame_df.sort_values(['video_id', 'timestamp_sec']).groupby('video_id', observed=True)['rsd_sec'].diff().dropna().max() <= 0 else 'review', 'RSD should never increase within a video')
add_check('media mounted', 'pass' if len(media_check) and media_check['exists'].any() else 'review', 'Set DATA_ROOT to inspect images/videos' if not len(media_check) or not media_check['exists'].any() else f"sample_exists={media_check['exists'].mean():.1%}")

quality_df = pd.DataFrame(checks)
display(quality_df)


## 13. Reading List

Start with these papers/resources for this project.

| Priority | Topic | Link | Why it matters |
|---:|---|---|---|
| 1 | Dataset | [MultiBypass140 GitHub](https://github.com/CAMMA-public/MultiBypass140) | Official dataset layout, labels, and frame extraction details. |
| 2 | Dataset / center shift | [Challenges in Multi-centric Generalization: Phase and Step Recognition in Roux-en-Y Gastric Bypass Surgery](https://arxiv.org/abs/2312.11250) | Core MultiBypass140 paper and cross-center motivation. |
| 3 | IAE detection | [Feature Mixing Approach for Detecting Intraoperative Adverse Events in Laparoscopic Roux-en-Y Gastric Bypass Surgery](https://arxiv.org/abs/2504.16749) | Relevant deviation/IAE detection baseline. |
| 4 | RSD | [RSDNet: Learning to Predict Remaining Surgery Duration from Laparoscopic Videos Without Manual Annotations](https://arxiv.org/abs/1802.03243) | Foundational remaining-duration formulation. |
| 5 | RSD | [Prediction of remaining surgery duration in laparoscopic videos based on visual saliency and the transformer network](https://doi.org/10.1002/rcs.2632) | Transformer-based RSD comparison point. |
| 6 | Cholec80 | [EndoNet](https://arxiv.org/abs/1602.03012) | Classic surgical phase/tool multi-task architecture and Cholec80 reference. |
| 7 | Temporal baseline | [TeCNO](https://arxiv.org/abs/2003.10751) | Strong online temporal convolutional phase-recognition baseline. |
| 8 | Transformer baseline | [SKiT](https://openaccess.thecvf.com/content/ICCV2023/papers/Liu_SKiT_a_Fast_Key_Information_Video_Transformer_for_Online_Surgical_ICCV_2023_paper.pdf) | Efficient online surgical video transformer. |
| 9 | HTA | [Surgformer](https://arxiv.org/abs/2408.03867) | Hierarchical temporal attention reference. |
| 10 | Surgical VLM | [HecVL](https://arxiv.org/abs/2405.10075) | Surgical video-language pretraining and zero-shot phase recognition. |
| 11 | Surgical VLM | [GP-VLS](https://arxiv.org/abs/2407.19305) | General-purpose surgical vision-language model. |
| 12 | Benchmark | [SurgBench](https://arxiv.org/abs/2506.07603) | Benchmark context for surgical video analysis. |
| 13 | Foundation model | [SurgMotion](https://arxiv.org/abs/2602.05638) | Recent video-native surgical foundation model direction. |


## 14. TL;DR

The most important figure for RSD is `figures/rsd_all_videos_linear.png`. It shows one linear RSD trajectory per video. The spread of starting heights is the duration variability the model must learn from visual and temporal context.


In [ ]:
print(f"""
Selected manifest: {LABEL_PATH.name}
Videos: {len(video_df):,}
Frames: {len(frame_df):,}
Centers: {dict(video_df['center'].value_counts())}
Splits: {dict(video_df['split'].value_counts())}
Duration median / mean / max: {video_df['duration_min'].median():.1f} / {video_df['duration_min'].mean():.1f} / {video_df['duration_min'].max():.1f} min
Deviation frames: {int(video_df['n_deviation_frames'].sum()):,} ({video_df['n_deviation_frames'].sum() / video_df['n_frames'].sum():.2%})
Deviation events: {len(event_df):,}
Phase-order clusters: {dict(video_df['phase_order_cluster'].value_counts().sort_index())}
Primary RSD figure: {FIG_DIR / 'rsd_all_videos_linear.png'}
""")
